# KG Gap Finder — local TriG test

Runs `kg_gap_finder.py` against a local `.trig` file, with no GraphDB / network dependency. Upload your `.trig` file and your `kg_gap_finder.py` script (or paste its contents into the cell below) and run top to bottom.

## 1. Install dependencies

In [2]:
!pip install -q rdflib SPARQLWrapper


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## 2. Upload your files
In case you run this on Colab.

Run this cell, then use the file picker to upload:
- your `.trig` file (the knowledge graph data)
- optionally, `kg_gap_finder.py` itself, if you'd rather import it as a module than paste its code below

In [ ]:
# from google.colab import files

# print("Select your .trig file (and kg_gap_finder.py if you have it as a separate file):")
# uploaded = files.upload()
# print("\nUploaded:", list(uploaded.keys()))

## 3. Set the trig filename

Update `TRIG_PATH` to match the filename you just uploaded.

In [1]:
TRIG_PATH = "mybrain_with_events.trig"  # <-- change this to match the uploaded filename

## 4. Load the graph

TriG encodes named graphs, so we use `rdflib.Dataset` (not a plain `Graph`) to preserve graph/context structure. If your gap finder logic doesn't care about named graphs and expects a flat triple store, you can swap this for `rdflib.Graph()` instead — but `Dataset` is the safer default for TriG.

In [2]:
import rdflib

g = rdflib.Dataset()
g.parse(TRIG_PATH, format="trig")

print(f"Loaded graph from {TRIG_PATH}")
print(f"Total quads: {len(g)}")
print(f"Named graphs: {[str(c.identifier) for c in g.graphs()]}")

Loaded graph from mybrain_with_events.trig
Total quads: 71
Named graphs: ['urn:x-rdflib:default']


## 5. Load kg_gap_finder

**Option A** — if you uploaded `kg_gap_finder.py` as a file in step 2, just import it:
```python
import kg_gap_finder
```

**Option B** — paste your script's code directly into the cell below (replacing the placeholder), if you'd rather not upload it as a separate file.

Either way: since we're bypassing GraphDB entirely, make sure any function that currently issues remote SPARQL queries via `SPARQLWrapper` has a local equivalent that runs `g.query(...)` directly against the `rdflib.Dataset` object loaded above.

In [3]:
import kg_gap_finder
import pprint

## 6. Run the gap analysis

In [4]:
untyped = kg_gap_finder.find_untyped_entities(graph=g)
dangling = kg_gap_finder.find_dangling_references(graph=g)
predicates = kg_gap_finder.find_predicate_gaps(graph=g, threshold=0.4)
objecttypes = kg_gap_finder.find_predicate_object_gaps(graph=g, threshold=0.4)
objectinstances = kg_gap_finder.find_predicate_object_gaps(graph=g, threshold=0.4)

print(f"Missing predicates: {len(predicates)}")
for s in predicates[:20]:
    print("  ", s)

print(f"Object types unknown: {len(objecttypes)}")
for s in objecttypes[:20]:
    print("  ", s)

print(f"Object instances unknown: {len(objectinstances)}")
for s in objectinstances[:20]:
    print("  ", s)

print(f"Untyped entities: {len(untyped)}")
for s in untyped[:20]:
    print("  ", s)

print(f"\nDangling references: {len(dangling)}")
for o in dangling[:20]:
    print("  ", o)

Missing predicates: 3
   {'class': 'http://cltl.nl/leolani/n2mu/Lunch', 'predicate': 'http://cltl.nl/leolani/n2mu/hasPatient', 'subject': 'http://cltl.nl/leolani/n2mu/lunch4', 'missing_count': 1, 'total_instances': 4, 'peer_coverage': '3/4'}
   {'class': 'http://cltl.nl/leolani/n2mu/Person', 'predicate': 'http://cltl.nl/leolani/n2mu/worksAt', 'subject': 'http://cltl.nl/leolani/n2mu/carl', 'missing_count': 1, 'total_instances': 3, 'peer_coverage': '2/3'}
   {'class': 'http://cltl.nl/leolani/n2mu/Wine', 'predicate': 'http://cltl.nl/leolani/n2mu/hasPortion', 'subject': 'http://cltl.nl/leolani/n2mu/wine3', 'missing_count': 1, 'total_instances': 3, 'peer_coverage': '2/3'}
Object types unknown: 8
   {'class': 'http://cltl.nl/leolani/n2mu/Lunch', 'predicate': 'http://cltl.nl/leolani/n2mu/hasPatient', 'object_type': 'http://cltl.nl/leolani/n2mu/Cheese', 'subject': 'http://cltl.nl/leolani/n2mu/lunch3', 'missing_count': 2, 'total_instances': 4, 'peer_coverage': '2/4'}
   {'class': 'http://cltl.n

## Use the report function to get all the results with a class filter

In [5]:
CLASSNAME="Event"
CLASSNAME="Entity"
#CLASSNAME="Consumable"
#CLASSNAME="Pizza"
#CLASSNAME="Drink"
CLASSNAME="Lunch"
class_filter="http://cltl.nl/leolani/n2mu/"+CLASSNAME

report = kg_gap_finder.build_report(graph=g, class_filter=class_filter, threshold=0.4)
for gaptype in report:
    print(gaptype, len(report[gaptype]))

print()

# gaptype = "untyped_entities"
# print(gaptype, len(report[gaptype]))
# for gap in report[gaptype]:
#     pprint.pp(gap)

#print()
# gaptype = "dangling_references"
# print(gaptype, len(report[gaptype]))
# for gap in report[gaptype]:
#     pprint.pp(gap)
#print()
gaptype = "predicate_gaps"
print(gaptype, len(report[gaptype]))
for gap in report[gaptype]:
    pprint.pp(gap)
print()

gaptype = "predicate_object_gaps"
print(gaptype, len(report[gaptype]))
for gap in report[gaptype]:
    pprint.pp(gap)
print()

# gaptype = "predicate_object_instances_gaps"
# print(gaptype, len(report[gaptype]))
# for gap in report[gaptype]:
#     pprint.pp(gap)

import json
f = f"thoughts_for_{CLASSNAME}_from_file_{TRIG_PATH}.json"
with open(f, "w") as f:
    json.dump(report, f, indent=2)
print(f"\nFull report written to {f}")

untyped_entities 3
predicate_gaps 1
dangling_references 6
predicate_object_gaps 5
predicate_object_instances_gaps 4

predicate_gaps 1
{'class': 'http://cltl.nl/leolani/n2mu/Lunch',
 'predicate': 'http://cltl.nl/leolani/n2mu/hasPatient',
 'subject': 'http://cltl.nl/leolani/n2mu/lunch4',
 'missing_count': 1,
 'total_instances': 4,
 'peer_coverage': '3/4'}

predicate_object_gaps 5
{'class': 'http://cltl.nl/leolani/n2mu/Lunch',
 'predicate': 'http://cltl.nl/leolani/n2mu/hasPatient',
 'object_type': 'http://cltl.nl/leolani/n2mu/Cheese',
 'subject': 'http://cltl.nl/leolani/n2mu/lunch3',
 'missing_count': 2,
 'total_instances': 4,
 'peer_coverage': '2/4'}
{'class': 'http://cltl.nl/leolani/n2mu/Lunch',
 'predicate': 'http://cltl.nl/leolani/n2mu/hasPatient',
 'object_type': 'http://cltl.nl/leolani/n2mu/Cheese',
 'subject': 'http://cltl.nl/leolani/n2mu/lunch4',
 'missing_count': 2,
 'total_instances': 4,
 'peer_coverage': '2/4'}
{'class': 'http://cltl.nl/leolani/n2mu/Lunch',
 'predicate': 'http:

## Report function with an empty class filter

In [13]:
report = kg_gap_finder.build_report(graph=g, class_filter="", threshold=0.4)
for gaptype in report:
    print(gaptype, len(report[gaptype]))

print()

# gaptype = "untyped_entities"
# print(gaptype, len(report[gaptype]))
# for gap in report[gaptype]:
#     pprint.pp(gap)

# gaptype = "dangling_references"
# print(gaptype, len(report[gaptype]))
# for gap in report[gaptype]:
#     pprint.pp(gap)

gaptype = "predicate_gaps"
print(gaptype, len(report[gaptype]))
for gap in report[gaptype]:
    pprint.pp(gap)

gaptype = "predicate_object_gaps"
print(gaptype, len(report[gaptype]))
for gap in report[gaptype]:
    pprint.pp(gap)

untyped_entities 8
predicate_gaps 8
dangling_references 8
predicate_object_gaps 8
predicate_object_instances_gaps 8

predicate_gaps 3
{'class': 'http://cltl.nl/leolani/n2mu/Lunch',
 'predicate': 'http://cltl.nl/leolani/n2mu/hasPatient',
 'missing_count': 1,
 'total_instances': 4,
 'peer_coverage': '3/4',
 'example_instances': ['http://cltl.nl/leolani/n2mu/lunch4']}
{'class': 'http://cltl.nl/leolani/n2mu/Person',
 'predicate': 'http://cltl.nl/leolani/n2mu/worksAt',
 'missing_count': 1,
 'total_instances': 3,
 'peer_coverage': '2/3',
 'example_instances': ['http://cltl.nl/leolani/n2mu/carl']}
{'class': 'http://cltl.nl/leolani/n2mu/Wine',
 'predicate': 'http://cltl.nl/leolani/n2mu/hasPortion',
 'missing_count': 1,
 'total_instances': 3,
 'peer_coverage': '2/3',
 'example_instances': ['http://cltl.nl/leolani/n2mu/wine3']}
predicate_object_gaps 6
{'class': 'http://cltl.nl/leolani/n2mu/Lunch',
 'predicate': 'http://cltl.nl/leolani/n2mu/hasPatient',
 'object_type': 'http://cltl.nl/leolani/n2m

## Notes

- Everything above runs entirely in-memory in this Colab VM — no GraphDB connection, no local network access needed.
- If your real `kg_gap_finder.py` currently talks to GraphDB via `SPARQLWrapper` for predicate-gap detection or other checks, port those queries to run via `g.query(...)` the same way as the placeholders above.
- If you later want to test against the *live* GraphDB instance from Colab instead of a static file, that requires GraphDB to be reachable over the internet (e.g. via ngrok or a public endpoint) — this notebook intentionally avoids that dependency.

# Using GraphDB endpoint

In [6]:
DBNAME= "event_sandbox"
DBNAME = "diabetes_event_details_and_types"
graphDB_endpoint = "http://localhost:7200/repositories/"+DBNAME
g = kg_gap_finder.load_graph_from_endpoint(graphDB_endpoint)
print(f"Total quads: {len(g)}")

Total quads: 135567


In [7]:
CLASSNAME="activity"
CLASSNAME="person"
#CLASSNAME="indoor"
#CLASSNAME="blood_sugar_levels"
classfilter="http://cltl.nl/leolani/n2mu/"+CLASSNAME

report = kg_gap_finder.build_report(graph=g, class_filter=classfilter, threshold=0.1)
# gaptype = "untyped_entities"
# print(gaptype, len(report[gaptype]))
# for gap in report[gaptype]:
#     pprint.pp(gap)

# gaptype = "dangling_references"
# print(gaptype, len(report[gaptype]))
# for gap in report[gaptype]:
#     pprint.pp(gap)
for gaptype in report:
    print(gaptype, len(report[gaptype]))
print()

gaptype = "predicate_gaps"
print(gaptype, len(report[gaptype]))
for gap in report[gaptype]:
    pprint.pp(gap)

gaptype = "predicate_object_gaps"
print(gaptype, len(report[gaptype]))
for gap in report[gaptype]:
    pprint.pp(gap)

f = f"thoughts_for_{CLASSNAME}_from_GraphDB_{DBNAME}.json"
with open(f, "w") as f:
    json.dump(report, f, indent=2)
print(f"\nFull report written to {f}")

untyped_entities 5
predicate_gaps 2
dangling_references 70
predicate_object_gaps 8
predicate_object_instances_gaps 0

predicate_gaps 2
{'class': 'http://cltl.nl/leolani/n2mu/person',
 'predicate': 'http://groundedannotationframework.org/gaf#denotedIn',
 'subject': 'http://cltl.nl/leolani/friends/piek-1',
 'missing_count': 2,
 'total_instances': 79,
 'peer_coverage': '77/79'}
{'class': 'http://cltl.nl/leolani/n2mu/person',
 'predicate': 'http://groundedannotationframework.org/gaf#denotedIn',
 'subject': 'http://cltl.nl/leolani/world/piek',
 'missing_count': 2,
 'total_instances': 79,
 'peer_coverage': '77/79'}
predicate_object_gaps 8
{'class': 'http://cltl.nl/leolani/n2mu/person',
 'predicate': 'http://groundedannotationframework.org/gaf#denotedIn',
 'object_type': 'http://groundedannotationframework.org/gaf#Mention',
 'subject': 'http://cltl.nl/leolani/friends/piek-1',
 'missing_count': 2,
 'total_instances': 79,
 'peer_coverage': '77/79'}
{'class': 'http://cltl.nl/leolani/n2mu/person'

In [21]:
report = kg_gap_finder.build_report(graph=g, class_filter="", threshold=0.4)
# gaptype = "untyped_entities"
# print(gaptype, len(report[gaptype]))
# for gap in report[gaptype]:
#     pprint.pp(gap)

# gaptype = "dangling_references"
# print(gaptype, len(report[gaptype]))
# for gap in report[gaptype]:
#     pprint.pp(gap)
for gaptype in report:
    print(gaptype, len(report[gaptype]))
print()

gaptype = "predicate_gaps"
print(gaptype, len(report[gaptype]))
for gap in report[gaptype]:
    pprint.pp(gap)

gaptype = "predicate_object_gaps"
print(gaptype, len(report[gaptype]))
for gap in report[gaptype]:
    pprint.pp(gap)

predicate_gaps 25
{'class': 'http://www.w3.org/2002/07/owl#Thing',
 'predicate': 'http://groundedannotationframework.org/gaf#denotedIn',
 'missing_count': 54,
 'total_instances': 119,
 'peer_coverage': '65/119',
 'example_instances': ['http://cltl.nl/leolani/friends/Jan',
                       'http://cltl.nl/leolani/friends/agent',
                       'http://cltl.nl/leolani/talk/chat0_utterance1_char0-96',
                       'http://cltl.nl/leolani/talk/chat0_utterance2_char0-128',
                       'http://cltl.nl/leolani/talk/chat0_utterance3_char0-264']}
{'class': 'http://groundedannotationframework.org/grasp#AttributionValue',
 'predicate': 'http://www.w3.org/2000/01/rdf-schema#label',
 'missing_count': 16,
 'total_instances': 31,
 'peer_coverage': '15/31',
 'example_instances': ['http://groundedannotationframework.org/grasp/emotion#anger',
                       'http://groundedannotationframework.org/grasp/emotion#disgust',
                       'http://groundedan


Full report written to <_io.TextIOWrapper name='thoughts.json' mode='w' encoding='UTF-8'>
